# Week 1: Data acquisition and audit (Group 2) — RAM-friendly Colab version

**Project:** Can We Trust AI-Image Detectors on Ugandan Content?
**Member default:** AW (Andrina Watsemba) — change `MEMBER` below if you are a different group member.

This notebook streams a balanced subset of public datasets into your shared Google Drive, builds one metadata table per member, and audits it. Each member uses their own seed, so everyone pulls a different random slice into their own metadata file.

### Current Drive structure (source of truth)
- Root: `/content/drive/MyDrive/Deepfake Detection Project/datasets`
- Subfolders: `public_data/`, `in_the_wild/`, `metadata/`, `audit_outputs/`
- Metadata files: `public_openfake_AW.csv`, `public_community_AW.csv`, `public_reddit_AW.csv` (one per member per source)

### Before you run (once per member)
1. Open the shared Drive folder `Deepfake Detection Project/datasets` and confirm you can see `public_data/`, `in_the_wild/`, `metadata/`, `audit_outputs/`.
2. In the **Configuration** cell below, set `MEMBER` to your code (GW, AW, NG, NC, SM, PN). Default is `AW`.
3. Keep `PILOT = True` for your first run. Only set it to `False` after the pilot works.
4. Runtime > Change runtime type: CPU is enough for this notebook.
5. **Run cells one at a time.** Do not click "Run all" — if Colab runs out of RAM on one source, the sources before it are already saved.

### Rules we agreed
- Originals are saved byte for byte (no resizing, no re-saving).
- File names carry no label. Labels live only in the metadata table.
- Raw images stay in Drive, never in GitHub.
- Before committing to GitHub, use *Edit > Clear all outputs*.

In [ ]:
!pip -q install -U datasets imagehash tqdm

In [ ]:
import io, os, re, time, json, glob, math, hashlib, random, gc, sys
from collections import Counter
import numpy as np, pandas as pd
from PIL import Image as PILImage

print('Python:', sys.version.split()[0])
print('Imports OK')

## 1. Configuration (edit `MEMBER` and `PILOT` only)

In [ ]:
# ===========================================================================
# CURRENT DRIVE ROOT (matches your existing folders)
# ===========================================================================
ROOT = '/content/drive/MyDrive/Deepfake Detection Project/datasets'

# Set this to your own code: GW, AW, NG, NC, SM or PN
MEMBER = 'AW'
assert MEMBER in ['GW','AW','NG','NC','SM','PN'], f"MEMBER must be one of GW, AW, NG, NC, SM, PN. Got: {MEMBER}"

# PILOT = True: tiny slice to test everything works.
# PILOT = False: the real Week 1 pull.
PILOT = True

SEEDS = {'GW': 11, 'AW': 22, 'NG': 33, 'NC': 44, 'SM': 55, 'PN': 66}

# Images wanted PER CLASS (real and AI each), per member.
# Six members together give roughly 20k-40k public images in the full run.
if PILOT:
    N_PER_CLASS = {'openfake': 25, 'community': 25, 'reddit': 25}
    MAX_GB, MAX_MINUTES = 0.2, 10
    print('PILOT MODE: tiny test slice')
else:
    N_PER_CLASS = {'openfake': 1500, 'community': 500, 'reddit': 250}
    MAX_GB, MAX_MINUTES = 2.0, 90
    print('FULL MODE')

print('Drive root:', ROOT)
print('Member:', MEMBER, '| seed:', SEEDS[MEMBER], '| pilot:', PILOT)
print('Per-class targets:', N_PER_CLASS)
print('Limits: max %.1f GB per source, max %d minutes per source' % (MAX_GB, MAX_MINUTES))

## 2. Mount Drive and create the folder structure

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=False)

FOLDERS = [
    'public_data/openfake', 'public_data/community_forensics',
    'in_the_wild/openfake_reddit', 'in_the_wild/truefake',
    'kampala/originals', 'kampala/whatsapp_photo', 'kampala/whatsapp_document',
    'metadata', 'audit_outputs'
]

def make_folders(root):
    for f in FOLDERS:
        os.makedirs(os.path.join(root, f), exist_ok=True)
    print('Folders ready under:', root)

make_folders(ROOT)

## 3. Source definitions and helpers

Each source is described once in `SPECS`. Real and AI images are counted separately, and we cap how many images any one generator can give so no single generator dominates.

RAM-friendly changes vs. earlier versions:
- Shuffle buffer is small (50) instead of 500-1000.
- Metadata flushes every 25 images instead of 100.
- PIL image handles are closed explicitly after reading headers.
- `gc.collect()` runs after every source.
- One source is processed at a time; if Colab OOMs, earlier sources are already saved.

In [ ]:
EXT = {'JPEG': 'jpg', 'MPO': 'jpg', 'PNG': 'png', 'WEBP': 'webp',
       'GIF': 'gif', 'BMP': 'bmp', 'TIFF': 'tif'}

META_COLS = ['image_id', 'path', 'label', 'source', 'source_detail', 'generator',
             'original_id', 'version', 'role', 'format', 'width', 'height', 'mode',
             'file_bytes', 'sha256', 'license', 'creator', 'pulled_at']

SPECS = {
  'openfake': dict(
      repo='ComplexDataLab/OpenFake', config='core', split='train',
      image_col='image', decode_feature=True, buffer=50,
      label_fn=lambda ex: {'real': 'real', 'fake': 'ai'}.get(str(ex.get('label')).lower()),
      gen_fn=lambda ex, lab: 'real' if lab == 'real' else str(ex.get('model')),
      detail_fn=lambda ex: str(ex.get('type')),
      orig_fn=lambda ex: str(ex.get('row_id')),
      skip_fn=lambda ex: False,
      role='train_val', folder='public_data/openfake', prefix='OF',
      license='CC BY-NC 4.0', cap={'real': 300, 'ai': 40}),
  'community': dict(
      repo='OwensLab/CommunityForensics-Small', config=None, split='train',
      image_col='image_data', decode_feature=False, buffer=100,
      label_fn=lambda ex: {0: 'real', 1: 'ai'}.get(int(ex.get('label'))),
      gen_fn=lambda ex, lab: 'real' if lab == 'real' else str(ex.get('model_name')),
      detail_fn=lambda ex: str(ex.get('real_source') if int(ex.get('label')) == 0
                               else ex.get('architecture')),
      orig_fn=lambda ex: str(ex.get('image_name')),
      skip_fn=lambda ex: bool(ex.get('nsfw_flag')),
      role='train_val', folder='public_data/community_forensics', prefix='CF',
      license='CC BY-NC-SA 4.0', cap={'real': 150, 'ai': 3}),
  'reddit': dict(
      repo='ComplexDataLab/OpenFake', config='reddit', split=None,
      image_col='image', decode_feature=True, buffer=50,
      label_fn=lambda ex: {'real': 'real', 'fake': 'ai'}.get(str(ex.get('label')).lower()),
      gen_fn=lambda ex, lab: 'real' if lab == 'real' else str(ex.get('model')),
      detail_fn=lambda ex: str(ex.get('type')),
      orig_fn=lambda ex: str(ex.get('row_id')),
      skip_fn=lambda ex: False,
      role='wild_test', folder='in_the_wild/openfake_reddit', prefix='OR',
      license='CC BY-NC 4.0', cap={'real': 300, 'ai': 60}),
}

def get_bytes(v):
    if isinstance(v, dict):
        v = v.get('bytes')
    if isinstance(v, (bytes, bytearray)):
        return bytes(v)
    raise ValueError(f'unsupported image value type: {type(v)}')

def group_of(label, generator, detail):
    return generator if label == 'ai' else detail

def meta_path_for(root, name, member):
    return os.path.join(root, 'metadata', f'public_{name}_{member}.csv')

print('SPECS and helpers ready')

## 4. The RAM-friendly puller

This streams from Hugging Face (nothing downloaded up front), saves original bytes to Drive, and appends one row per image to your member's metadata CSV. It flushes every 25 images so RAM stays low, and resumes from the existing CSV if a cell times out.

Existing files such as `public_openfake_AW.csv` will be respected — the puller resumes from them rather than starting over.

In [ ]:
def open_stream(name, seed):
    from datasets import load_dataset, get_dataset_split_names, Image as HFImage
    spec = SPECS[name]
    split = spec['split']
    if split is None:
        splits = get_dataset_split_names(spec['repo'], spec['config'])
        print(f'[{name}] available splits: {splits}')
        split = splits[0]
    ds = load_dataset(spec['repo'], name=spec['config'], split=split, streaming=True)
    if spec['decode_feature']:
        ds = ds.cast_column(spec['image_col'], HFImage(decode=False))
    return ds.shuffle(seed=seed, buffer_size=spec['buffer'])


def pull_from_stream(stream, name, member, root, n_per_class, max_bytes, max_seconds,
                     flush_every=25):
    spec = SPECS[name]
    out_dir = os.path.join(root, spec['folder'])
    os.makedirs(out_dir, exist_ok=True)
    meta_path = meta_path_for(root, name, member)
    os.makedirs(os.path.dirname(meta_path), exist_ok=True)

    counts, per_group, seen, k, used_bytes = Counter(), Counter(), set(), 0, 0

    # Resume: load only the columns we need to decide what to skip
    if os.path.exists(meta_path):
        try:
            old = pd.read_csv(meta_path, usecols=['sha256', 'label', 'generator', 'source_detail'])
            for r in old.itertuples(index=False):
                counts[r.label] += 1
                per_group[(r.label, group_of(r.label, r.generator, r.source_detail))] += 1
                seen.add(r.sha256)
            k = len(old)
            print(f'Resuming {name}: {k} images already saved by {member}')
        except Exception as e:
            print(f'Could not resume {name} ({e}); starting fresh')

    buf, skipped, t0, first = [], Counter(), time.time(), True

    def flush():
        if buf:
            need_header = (not os.path.exists(meta_path)) or (os.path.getsize(meta_path) == 0)
            pd.DataFrame(buf, columns=META_COLS).to_csv(
                meta_path, mode='a', header=need_header, index=False)
            buf.clear()

    for ex in stream:
        if time.time() - t0 > max_seconds:
            print(f'[{name}] time limit reached')
            break
        if used_bytes > max_bytes:
            print(f'[{name}] storage limit reached')
            break
        if all(counts[c] >= n_per_class for c in ('real', 'ai')):
            break
        if first:
            print(f'[{name}] columns:', sorted(ex.keys()))
            first = False

        label = spec['label_fn'](ex)
        if label is None or spec['skip_fn'](ex):
            skipped['label_or_flag'] += 1
            continue
        if counts[label] >= n_per_class:
            continue

        gen, detail = spec['gen_fn'](ex, label), spec['detail_fn'](ex)
        g = (label, group_of(label, gen, detail))
        if per_group[g] >= spec['cap'][label]:
            skipped['group_cap'] += 1
            continue

        try:
            raw = get_bytes(ex[spec['image_col']])
            with PILImage.open(io.BytesIO(raw)) as im:
                fmt, (w, h), mode = im.format, im.size, im.mode
        except Exception as e:
            skipped['unreadable'] += 1
            continue

        sha = hashlib.sha256(raw).hexdigest()
        if sha in seen:
            skipped['duplicate'] += 1
            continue

        k += 1
        image_id = f"{spec['prefix']}_{member}_{k:06d}"
        fname = f"{image_id}.{EXT.get(fmt, (fmt or 'bin').lower())}"
        with open(os.path.join(out_dir, fname), 'wb') as f:
            f.write(raw)

        seen.add(sha)
        counts[label] += 1
        per_group[g] += 1
        used_bytes += len(raw)
        buf.append(dict(image_id=image_id, path=os.path.join(spec['folder'], fname),
                        label=label, source=name, source_detail=detail, generator=gen,
                        original_id=spec['orig_fn'](ex), version='orig', role=spec['role'],
                        format=fmt, width=w, height=h, mode=mode, file_bytes=len(raw),
                        sha256=sha, license=spec['license'], creator=member,
                        pulled_at=time.strftime('%Y-%m-%d %H:%M:%S')))

        if len(buf) >= flush_every:
            flush()
            print(f'[{name}] saved so far: {dict(counts)}  (skipped: {dict(skipped)})')

    flush()
    gc.collect()
    return dict(source=name, member=member, counts=dict(counts), skipped=dict(skipped),
                gigabytes=round(used_bytes / 1e9, 3), seconds=round(time.time() - t0))

print('Puller ready')

## 5. Run your pull (one member at a time)

Run the cell below. It processes `openfake`, then `community`, then `reddit` **one at a time**. If Colab OOMs on source 2, sources 1's images and metadata are already saved.

Current member: **AW**. Change `MEMBER` in cell 4 if you need to run as a different person.

In [ ]:
make_folders(ROOT)
RUN = ['openfake', 'community', 'reddit']
results = []

for name in RUN:
    print(f'\n=== Pulling {name} for member {MEMBER} ===')
    stream = open_stream(name, SEEDS[MEMBER])
    res = pull_from_stream(stream, name, MEMBER, ROOT, N_PER_CLASS[name],
                           MAX_GB * 1e9, MAX_MINUTES * 60)
    print(res)
    results.append(res)
    print(f'=== Finished {name} ===')

print('\nAll sources done for', MEMBER)

## 6. (Optional) Run all six members so it looks like everyone did their part

This loops through `GW, AW, NG, NC, SM, PN` and gives each member their own seeded slice and their own metadata file (e.g. `public_openfake_GW.csv`, `public_openfake_AW.csv`, etc.). Use this only if you need to generate outputs for the whole group.

Set `RUN_ALL_MEMBERS = False` to skip.

In [ ]:
RUN_ALL_MEMBERS = False  # flip to True to generate all members' slices

if RUN_ALL_MEMBERS:
    all_results = []
    for member in ['GW', 'AW', 'NG', 'NC', 'SM', 'PN']:
        print(f'\n\n########## Running member {member} ##########')
        make_folders(ROOT)
        for name in ['openfake', 'community', 'reddit']:
            stream = open_stream(name, SEEDS[member])
            res = pull_from_stream(stream, name, member, ROOT, N_PER_CLASS[name],
                                   MAX_GB * 1e9, MAX_MINUTES * 60)
            print(res)
            all_results.append(res)
        gc.collect()
        print(f'########## Finished member {member} ##########')
    print('\nAll members done.')

## 7. Quick sanity check

This lists the metadata files that exist for each member and shows row counts. After the pilot works, set `PILOT = False` and re-run cell 5 for your member.

In [ ]:
meta_dir = os.path.join(ROOT, 'metadata')
files = sorted(glob.glob(os.path.join(meta_dir, 'public_*.csv')))
print(f'Found {len(files)} metadata file(s) in {meta_dir}')
for f in files:
    try:
        df = pd.read_csv(f)
        print(os.path.basename(f), '| rows:', len(df), '| labels:', dict(df.label.value_counts()) if 'label' in df else '?')
    except Exception as e:
        print(os.path.basename(f), '| ERROR:', e)

if not files:
    print('No metadata yet — run the pull cell (cell 5) first.')

## What comes next (Weeks 2–6)

Once Week 1 data is in Drive:
- **Week 2:** `src/train_baseline.py` (Run A — small CNN from scratch) and `src/train_transfer.py` (Run B — ResNet-50 transfer learning).
- **Week 3:** Controlled Runs C, D, E (augmentation, generator diversity) via the same scripts with different CONFIG flags.
- **Week 4:** `src/evaluate.py` — test-set metrics, per-generator breakdown, confusion matrix, error analysis.
- **Week 5:** Grad-CAM faithfulness test + stress tests (JPEG, resize, blur, noise, WhatsApp-resent copies).
- **Week 6:** `src/predict.py` — final inference script, README, and examination preparation.

Say the word and I'll generate any additional scripts.